# Protoype - Building a Vector Search DB Based on an existing Chunk Table

## Overview
Simple vector search solution for document retrieval. 
- Transforms document chunks into semantic vectors. 
- Create a vector search index. 
- Use advanced(?) Search and re-ranking to boost retrieval precision. 

The sample chunks are small and mostly encompass one document per chunk due to the nature of the prototype source table, which consists of short reviews of franchises. 
Document chunking is not covered in this protoype. 

In real-world retrieval scenarios, converting unstructured documents into searchable semantic vectors enables more accurate and context-aware results. This workflow empowers you to efficiently find relevant information, even when keywords don’t match exactly.

## Requirements
- The `samples.bakehouse.media_gold_reviews_chunked` sample table.
- **Serverless Compute (environment version 5)**. Follow the instructions [here](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) to select the appropriate environment version.
- Access to Foundation Model APIs for embedding generation.
- Appropriate permissions to create and manage vector search indexes.

%md
## A. Prepare Source Table

- Vector Search requires the source table to have Change Data Feed (CDF) enabled. Verify this property is enabled. 
- A unique identifier per chunk is also needed to create the vector search index. 

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()                                                                                                                 


In [0]:
reviews_chunked = "samples.bakehouse.media_gold_reviews_chunked"
catalog = "data_studio_interviews" #SHOULD CREATE NEW ONE, CAN'T
schema = "default"

In [0]:
import os
def is_serverless_5():
    is_serverless = os.environ.get('IS_SERVERLESS', '')
    runtime_version = os.environ.get('DATABRICKS_RUNTIME_VERSION', '')
    return is_serverless == 'TRUE' and runtime_version.startswith('client.5.')

is_serverless_5()

In [0]:
# Verify if Change Data Feed is enabled on the table
props = spark.sql(f"SHOW TBLPROPERTIES {reviews_chunked}").collect()
props_dict = {row['key']: row['value'] for row in props}

cdf_enabled = props_dict.get('delta.enableChangeDataFeed', 'false') == 'true'
if cdf_enabled:
    print(f"Change Data Feed is enabled on `{reviews_chunked}`")
else:
    print(f"Change Data Feed is NOT enabled on `{reviews_chunked}`")

# Display sample data to understand table structure
display(spark.sql(f"SELECT * FROM {reviews_chunked} LIMIT 5"))


## B. Computing Embeddings for Document Chunks

Databricks supports two main approaches for generating embeddings:
* **Managed embeddings:** Vector Search automatically computes and manages embeddings for you, simplifying setup and maintenance. This is the recommended approach for most use cases.

* **Manual embeddings:** You can generate embeddings externally (using MLflow Deployments, Hugging Face, OpenAI, etc.) and store them in a column. For large datasets, you can use a Spark UDF to compute embeddings for each row in a text column.

In this demo, we will use **managed embeddings**, allowing Vector Search to compute and maintain the embeddings for us.

In [0]:
import mlflow.deployments

# Initialize deployment client for accessing embedding models
deploy_client = mlflow.deployments.get_deploy_client("databricks")

# Generate embeddings for a sample question
sample = "The temple monks conspired with Judas Iscariot to quell threats of rebellion against the church"
response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [sample]})
embeddings = [e["embedding"] for e in response.data]

# Display embedding information
print("Embedding for sample:", embeddings[0])
print("Embedding shape:", len(embeddings[0]))

### C. Creating Index via SDK

A vector search index can be created via the SDK. 
Using the embedding tested in the previous cell. 

[Vector Search SDK documentation](https://api-docs.databricks.com/python/vector-search/index.html).

**Note:** Index refresh modes can be set to manual or sync, depending on your update requirements.


In [0]:
vector_search_endpoint = "vs-proto-endpoint"

from databricks.vector_search.client import VectorSearchClient

# Initialize the Vector Search client
vsc = VectorSearchClient(disable_notice=True)
index_name = f"{catalog}.{schema}.docs_chunked_index"


In [0]:
#Create endpoint CONSTANT COST --MUST BE DELETED AFTERWARDS
vsc.create_endpoint_and_wait(name=vector_search_endpoint, endpoint_type="STANDARD")

In [0]:
# Define index name using three-level naming convention
index_name = f"{catalog}.{schema}.docs_chunked_index"

# Create the index using managed embeddings with Delta Sync
vsc.create_delta_sync_index_and_wait(
    endpoint_name=vector_search_endpoint,
    index_name=index_name,
    source_table_name=reviews_chunked,
    primary_key='franchiseID',
    embedding_source_column="chunk_id",
    embedding_model_endpoint_name="databricks-gte-large-en",
    pipeline_type="TRIGGERED",
)
print(f"Index '{index_name}' created for table '{reviews_chunked}' using endpoint '{vector_search_endpoint}'.")

## D. Search Methods: Query, Hybrid, and Full-Text
- **Query Search:** Uses embeddings to find semantically similar chunks.
- **Hybrid Search:** Combines semantic and keyword-based search for improved relevance.
- **Full-Text Search:** Retrieves chunks based on exact keyword matches. Not enabled in this workspace.

Results can also be filtered by the `path` field to target specific documents, but in this case it is not relevant, since the table was pre-chunked and did not originate from a document store. 

In [0]:
# Get the vector search index for performing searches
index = vsc.get_index(index_name=index_name)
print(index.describe())

### D1. Query Search: Similarity Search

In [0]:
query_text = "Return negative experiences in Asia"
results = index.similarity_search(
    query_text=query_text,
    columns=["chunk_id", "chunked_text"],
    num_results=3
)
display(results)

### D2. Hybrid Search: Semantic + Keyword

In [0]:
query_text = "Provide explicitly 1/5 star experiences in Tokyo"
results_hybrid = index.similarity_search(
    query_text=query_text,
    columns=["chunk_id", "chunked_text"],
    query_type="hybrid",
    num_results=5
)
display(results_hybrid)

## E. Re-Ranking

As shown above, vector search Databases often return similar but semantically irrelevant chunks. These are close in wording but weak in context. 
Reranking improves precision(# of Relevant results / total results).
It re-evaluates the top results using another model or addiotional signals. 

The main drawback is increased compute cost per query. Significantly improves accuracy. 
Should be used for high value queries, and avoided for less critical use cases. 
Use selectively to balance performance and precision.

In [0]:
from databricks.vector_search.reranker import DatabricksReranker

query_text = "Provide explicitly 1/5 star experiences in Tokyo"
results_reranked = index.similarity_search(
    query_text=query_text,
    columns=["chunk_id", "chunked_text"],
    num_results=5,
    query_type="hybrid",
    reranker=DatabricksReranker(columns_to_rerank=["chunked_text"])
)

display(results_reranked)

### F. Cleaning Up
Vector Search Endpoints are billed hourly and always on. 
One VS endpoint can support multiple indexes [over 50!](https://docs.databricks.com/aws/en/vector-search/vector-search-cost-management), so they are static in cost in this regard, however, they cannot be turned off or paused, so they must be eliiminated for testing purposes. 




In [0]:
# Cleanup — index must be deleted before endpoint
vsc.delete_index(index_name=index_name)
vsc.delete_endpoint(name=vector_search_endpoint)                              

In [0]:
# Verify deletion                                                           
index_gone = not vsc.index_exists(index_name=index_name)
endpoint_gone = not vsc.endpoint_exists(name=vector_search_endpoint) 

if index_gone and endpoint_gone:
    print(f"Index '{index_name}' and endpoint '{vector_search_endpoint}'deleted successfully ")                                                       
else:
    if not index_gone:                                                        
        print(f"WARNING: Index '{index_name}' still exists.")               
    if not endpoint_gone:
        print(f"WARNING: Endpoint '{vector_search_endpoint}' still exists.")